In [60]:
!git clone https://github.com/naj2139/C6998-2026fall-assignment2-naj2139.git
%cd C6998-2026fall-assignment2-naj2139/HPML_HW2_starter
!pip install -q transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 otter-grader==6.1.6

Cloning into 'C6998-2026fall-assignment2-naj2139'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 109 (delta 38), reused 75 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 261.94 KiB | 9.70 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/C6998-2026fall-assignment2-naj2139/HPML_HW2_starter/C6998-2026fall-assignment2-naj2139/HPML_HW2_starter/C6998-2026fall-assignment2-naj2139/HPML_HW2_starter


In [61]:
# Initialize Otter
import otter
grader = otter.Notebook("master.ipynb")

**Environment**

| Component | Version / Details |
|---|---|
| Platform | Google Colab |
| GPU | NVIDIA Tesla T4 (Turing, compute capability 7.5, 40 SMs, 15.6 GB memory) |
| CPU | 2 vCPUs |
| CUDA runtime / driver API | 13.0 / 13.0 (CUPTI 13.0.1) |
| Python | 3.13 |
| PyTorch | 2.11.0+cu130 |
| transformers | 4.46.3 |
| datasets | 3.1.0 |
| accelerate | 1.1.1 |
| otter-grader | 6.1.6 |

# HW2: Profiling and Accelerating a Small LLM Workload

**COMS 6998 -- High Performance Machine Learning, Columbia University**

**Instructor:** Dr. Kaoutar El Maghraoui

**Weeks covered:** 3 and 4 (PyTorch Runtime and Profiling; PyTorch Performance and `torch.compile`)

**Due:** October 25, 2026

**Points:** 100 (+10 extra credit)

A small set of functions you write is autograded on CPU against a tiny model and
a 300-row IMDB sample (fast, offline). The experiments (profiler traces,
compile, mixed precision, sweeps) need a GPU and are graded from your
plots, tables, traces, and written analysis. GPU work is gated behind
`FULL_SCALE = torch.cuda.is_available()` so the notebook runs top to bottom on a
CPU grading container. Do not remove the `FULL_SCALE` gates.

### Tasks
- C1 (12): fine-tune DistilBERT with your own loop.
- C2 (8): per-step timing breakdown with correct CUDA sync.
- C3 (10): input pipeline, num_workers sweep plus pinning and copy overlap.
- C4 (15): profiler trace; classify the step as launch-bound, compute-bound, or input-bound, with evidence.
- C5 (10): find, count, and fix graph breaks.
- C6 (12): torch.compile performance and the compile break-even point.
- C7 (10): mixed precision; step-time and peak-memory change.
- C8 (8): optimizers compared on time, convergence, and memory.
- Q1-Q6 (15): short answers. EC1-EC2 (+10): hidden sync; recompilation.

## Setup

Run this cell first. It picks the device, loads the tokenizer from the
bundled `tiny_tokenizer/` folder (the exact same vocabulary as
`distilbert-base-uncased`, so everything you learn here transfers directly),
and sets a few constants used throughout the notebook.

On your GPU machine you can instead do
`AutoTokenizer.from_pretrained("distilbert-base-uncased")` -- it is the same
tokenizer, just fetched fresh from the Hub. We ship a local copy so the
notebook (and the autograder) never depends on network access.

In [62]:
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, DistilBertConfig, DistilBertForSequenceClassification

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FULL_SCALE = torch.cuda.is_available()  # True on your GPU box, False under CPU grading

DATA_DIR = Path("data")
TOKENIZER_DIR = Path("tiny_tokenizer")

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)

print(f"device: {DEVICE}, FULL_SCALE: {FULL_SCALE}")
print(f"tokenizer vocab_size: {tokenizer.vocab_size}, pad_token_id: {tokenizer.pad_token_id}")

device: cuda, FULL_SCALE: True
tokenizer vocab_size: 30522, pad_token_id: 0


## Data

`data/tiny_imdb.jsonl` is a 300-row sample drawn from the real IMDB training
split (150 positive, 150 negative, `{"text": ..., "label": ...}` per line),
bundled so the notebook and the autograder work fully offline. Use it for
every correctness check below and for quick local iteration.

For the actual graded experiments (C1's full run, C2-C8) you load the real,
full IMDB dataset with `datasets.load_dataset("imdb")` on your GPU machine.
That full-scale loading is inside the `FULL_SCALE` branches further down so
it never runs during CPU grading.

In [63]:
def load_tiny_imdb(path=DATA_DIR / "tiny_imdb.jsonl"):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

tiny_rows = load_tiny_imdb()
print(f"tiny_imdb: {len(tiny_rows)} rows, label balance: "
      f"{sum(r['label'] for r in tiny_rows)} positive / "
      f"{sum(1 - r['label'] for r in tiny_rows)} negative")

tiny_imdb: 300 rows, label balance: 150 positive / 150 negative


## The tiny model config used for grading

The functions you write below are tested against a **tiny, randomly
initialized DistilBERT** (2 layers, hidden size 32) built from a
`DistilBertConfig` -- not a full pretrained model. It shares the real
tokenizer's vocabulary (so `vocab_size` and special-token ids match
`distilbert-base-uncased` exactly), which is what makes Q1-Q4 below
meaningful, but it has ~1M parameters instead of ~67M and trains in a
fraction of a second per step. This is the model the hidden tests use; it is
provided, not something you need to write.

In [64]:
def build_tiny_config(num_labels=2, n_layers=2, dim=32, n_heads=4, hidden_dim=64):
    """A tiny DistilBertConfig: same vocabulary as distilbert-base-uncased,
    far fewer layers/dimensions, for fast CPU-only correctness checks."""
    return DistilBertConfig(
        vocab_size=tokenizer.vocab_size,
        max_position_embeddings=128,
        sinusoidal_pos_embds=False,
        n_layers=n_layers,
        n_heads=n_heads,
        dim=dim,
        hidden_dim=hidden_dim,
        dropout=0.1,
        attention_dropout=0.1,
        pad_token_id=tokenizer.pad_token_id,
        num_labels=num_labels,
    )


def build_model(config):
    """Build a DistilBertForSequenceClassification from a config. On your GPU
    run, for the real fine-tuning experiments, you'll instead call
    AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased',
    num_labels=2) to get pretrained weights -- build_model here just wraps
    DistilBertForSequenceClassification(config) so the same call shape works
    for both the tiny and the full-scale model."""
    return DistilBertForSequenceClassification(config)


TINY_CONFIG = build_tiny_config()
tiny_model = build_model(TINY_CONFIG)
print(tiny_model.config)
print(f"tiny model parameter count: {sum(p.numel() for p in tiny_model.parameters()):,}")

DistilBertConfig {
  "_attn_implementation_autoset": true,
  "activation": "gelu",
  "attention_dropout": 0.1,
  "dim": 32,
  "dropout": 0.1,
  "hidden_dim": 64,
  "initializer_range": 0.02,
  "max_position_embeddings": 128,
  "model_type": "distilbert",
  "n_heads": 4,
  "n_layers": 2,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "transformers_version": "4.46.3",
  "vocab_size": 30522
}

tiny model parameter count: 999,074


## C1 (12 pts): Fine-tune a small LLM with your own loop

### Tokenization and dynamic padding

Write `tokenize_function(text, tokenizer, max_len)`: tokenize a single
review's text, **truncating** to at most `max_len` tokens. Do **not** pad
here -- padding to a fixed length up front wastes compute on short reviews.
Instead, `make_collate_fn(tokenizer)` returns a `collate_fn(batch)` that pads
**dynamically**, only up to the longest example in each batch.

In [65]:
def tokenize_function(text, tokenizer, max_len=256):
    """
    Tokenize a single review's text into a list of token ids.
    Truncate to at most `max_len` tokens. Do NOT pad here.

    Args:
        text: a single review string.
        tokenizer: a Hugging Face tokenizer.
        max_len: maximum number of tokens to keep (including special tokens).

    Returns:
        A list[int] of token ids.
    """
    # ---- Inspect tokenizer output: tokenizer(text) ----
    # output = tokenizer(text)

    # BatchEncoding is a subclass of Python's dict
    # print("Type: ", type(output))           # <class 'transformers.tokenization_utils_base.BatchEncoding'>
    # print("Dict Keys :", output.keys())     # dict_keys(['input_ids', 'attention_mask'])

    # Show all arguments accepted by tokenizer(text, ...)
    # help(tokenizer.__call__)

    # token IDs == input_ids
    return tokenizer(text=text, truncation=True, padding=False, max_length=max_len)['input_ids']

# ---- Sanity checks: inspect raw data format and the tokenizer from Setup ----
# VAR: tiny_row
# print("Type: ", type(tiny_rows)) # List[Dict]
# print("First Line: ", tiny_rows[0]['text']) # label is already an int, so no tokenization needed; keep it as-is

# Tokenizer loaded in the Setup section
# tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)

text = tiny_rows[0]['text']
tokenized_text = tokenize_function(text, tokenizer)

**Your turn.** Implement `make_collate_fn(tokenizer)`. It must return a
function `collate_fn(batch)` where `batch` is a list of dicts
`{"input_ids": List[int], "label": int}`. `collate_fn` should:

- pad every example's `input_ids` to the length of the **longest** example in
  the batch, using `tokenizer.pad_token_id`;
- build a matching `attention_mask` (1 for real tokens, 0 for padding);
- stack `label` into a `LongTensor`.

Return a dict `{"input_ids": LongTensor[B, T], "attention_mask": LongTensor[B, T], "labels": LongTensor[B]}`.

In [66]:
# ---- Closure: make_collate_fn builds collate_fn ----
# Factory (closure): runs once, captures pad_id, and returns the inner collate_fn
# The returned collate_fn is then called with a batch: collate_fn(batch)

def make_collate_fn(tokenizer):
    # pad_id = token ID of [PAD] (0 for this tokenizer) -> it's the FILL VALUE, not a padding amount
    pad_id = tokenizer.pad_token_id

    def collate_fn(batch):

        # ---- Sanity checks: inspect the batch passed into collate_fn ----
        # VAR: batch
        # print("Type: ", type(batch)) List[Dict]
        # print("Length: ", len(batch)) # 4
        # batch is a LIST of dicts, so batch["label"] doesn't work -> loop over examples

        # ---- Padding target: longest sequence in this batch ----
        max_len = max(len(data['input_ids']) for data in batch)

        # ---- Per-example: convert to tensors, build mask, pad ----
        # Lists collect each example's tensors; stacked into batched tensors after the loop
        # Output key is "labels" (plural) per the spec; input dicts use "label"
        stack_data = {'input_ids': [], 'labels': [], 'attention_mask': []}
        for data in batch:
            # Use LOCAL variables, never write back into `data`:
            # mutating the caller's dicts broke the test (it re-reads batch after collate,
            # saw every sequence already padded to max_len, and picked the wrong "longest" row)

            # torch.tensor (lowercase) = factory function, accepts dtype=
            # torch.Tensor (capital) = legacy class constructor, always float32, NO dtype= kwarg
            # torch.int64 counts as a LongTensor
            input_ids = torch.tensor(data['input_ids'], dtype=torch.int64)
            label = torch.tensor(data['label'], dtype=torch.int64)

            # Attention mask: 1 for real tokens, 0 for padding (built before padding,
            # so len(input_ids) is still the original length)
            # torch.cat on 1-D tensors -> default dim=0 (dim=1 doesn't exist for 1-D)
            ones = torch.ones(len(input_ids), dtype=torch.int64)
            zero = torch.zeros(max_len - len(input_ids), dtype=torch.int64)
            attention_mask = torch.cat((ones, zero))

            # Right-pad input_ids up to max_len, filling with the tokenizer's pad token
            # pad=(left, right) = HOW MANY positions to add, not the target length
            # -> right amount must be per-sequence: max_len - len(input_ids)
            # (nn.ZeroPad1d always fills 0; F.pad lets value=pad_id be explicit)
            input_ids = nn.functional.pad(
                input=input_ids,
                pad=(0, max_len - len(input_ids)),
                value=pad_id,
            )

            # Collect this example's tensors (all same length now, so they can be stacked)
            stack_data['input_ids'].append(input_ids)
            stack_data['labels'].append(label)
            stack_data['attention_mask'].append(attention_mask)

        # ---- Stack: list of per-example tensors -> one batched tensor per field ----
        # Can't just return batch: the model needs ONE (B, T) tensor per field for a single
        # parallel forward pass, not a list of B separate tensors
        # Stack once after the loop (stacking inside would copy the growing tensor every step)
        # This loop is 3 stack CALLS (one per field), not re-looping over examples
        # input_ids, attention_mask: (batch_size, max_len) | labels: (batch_size,)
        for key in stack_data:
            stack_data[key] = torch.stack(stack_data[key])

        return stack_data

    return collate_fn

collate_fn = make_collate_fn(tokenizer)
try:
    _demo_batch = [
        {"input_ids": tokenize_function(r["text"], tokenizer, max_len=64), "label": r["label"]}
        for r in tiny_rows[:4]
    ]
    _demo_out = collate_fn(_demo_batch)
    print({k: v.shape for k, v in _demo_out.items()})
except Exception as _e:
    print(f"(demo skipped -- implement make_collate_fn above: {type(_e).__name__}: {_e})")

{'input_ids': torch.Size([4, 64]), 'labels': torch.Size([4]), 'attention_mask': torch.Size([4, 64])}


In [67]:
grader.check("q_collate")

q_collate results: All test cases passed!

### The training step

Write `train_one_step(model, batch, optimizer, device)`: move the batch to
`device`, run a forward pass, compute the loss, backpropagate, and take one
optimizer step. Return the scalar loss as a Python `float`.

In [68]:
def train_one_step(model, batch, optimizer, device):
    """
    Run a single training step and return the loss as a Python float.

    Args:
        model: a DistilBertForSequenceClassification.
        batch: a dict with "input_ids", "attention_mask", "labels" tensors
            (as produced by collate_fn), still on CPU.
        optimizer: a torch.optim.Optimizer already built over model.parameters().
        device: the torch.device to run on.

    Steps: zero the gradients, move the batch to `device`, forward pass with
    labels (so the model computes the loss for you), backward pass, optimizer
    step. Return loss.item().
    """

    # ---- Zero gradients ----
    # Clear gradients from the previous step -> PyTorch ACCUMULATES grads in .grad by default,
    # so without this, each backward() adds to the old gradients instead of replacing them
    optimizer.zero_grad()

    # ---- Move model to GPU ----
    # nn.Module.to() is in-place (moves parameters, returns the same model),
    # so `model.to(device)` alone works; `model = ...` is optional but harmless
    model = model.to(device)

    # ---- Move batch to GPU ----
    # batch is a dict (from collate_fn), and dicts have no .to() -> only tensors do
    # tensor.to() is NOT in-place: it returns a new GPU tensor, so the result must be used/stored
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    # ---- Forward pass ----
    # pred is NOT a tensor -> it's an output object (SequenceClassifierOutput)
    # bundling tensors as attributes: pred.logits [B, num_labels], and pred.loss when labels are passed
    # Can't reshape/pass pred itself to a criterion -> the tensor a loss needs is pred.logits
    pred = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

    # Manual check: CrossEntropyLoss takes logits [B, num_labels] + integer labels [B] directly,
    # (BCEWithLogitsLoss doesn't fit: it wants float targets shaped like logits, not integer class labels [B])
    # criterion = nn.CrossEntropyLoss()
    # loss = criterion(pred.logits, labels)
    # print(loss)

    # ---- Get loss ----
    # Passing labels makes the model compute cross-entropy loss internally -> pred.loss
    # Must assign it: pred.loss on its own line just reads the value and discards it
    loss = pred.loss

    # ---- Backward pass + update ----
    # loss.backward(): backprop through the graph, computing d(loss)/d(param) for every
    # parameter and storing it in param.grad
    # optimizer.step(): uses those .grad values to update the parameters (e.g. AdamW update rule)
    # Order matters: zero_grad -> forward -> backward -> step
    loss.backward()
    optimizer.step()

    # ---- Return loss as Python float ----
    # loss is a 0-dim tensor (still on GPU, attached to the graph)
    # .item() copies its single value to CPU as a plain Python float
    # .to(torch.float32) only changes dtype -> still a tensor, so it doesn't satisfy the spec
    return loss.item()


In [69]:
grader.check("q_train_step")

q_train_step results: All test cases passed!

### The full fine-tuning run

On your GPU machine: load the **real** IMDB dataset (`datasets.load_dataset("imdb")`),
tokenize and collate it with the functions above, fine-tune
`distilbert-base-uncased` (`AutoModelForSequenceClassification.from_pretrained`)
for 5 epochs, and report **training loss and training accuracy per epoch**.
Make sure the same code path also runs on CPU (it will just be slow) --
don't special-case CUDA in your training loop itself, only in *how much* data
you run it on.

Report your per-epoch loss/accuracy as a table or a plot, whichever you
prefer.

In [ ]:
# C1: the real fine-tuning run on GPU. On CPU this stays tiny.
from torch.utils.data import Dataset, DataLoader

class IMDBDataset(Dataset):
    def __init__(self, rows, tokenizer, max_len=256):
        self.rows, self.tokenizer, self.max_len = rows, tokenizer, max_len
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return {"input_ids": tokenize_function(r["text"], self.tokenizer, self.max_len),
                "label": r["label"]}

if FULL_SCALE:
    from transformers import AutoModelForSequenceClassification
    from datasets import load_dataset
    imdb = load_dataset("stanfordnlp/imdb")
    train_rows = [{"text": t, "label": l} for t, l in
                  zip(imdb["train"]["text"][:2000], imdb["train"]["label"][:2000])]
    c1_model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2).to(DEVICE)
    train_loader = DataLoader(IMDBDataset(train_rows, tokenizer), batch_size=32,
                              shuffle=True, collate_fn=collate_fn, num_workers=2)
    c1_optimizer = torch.optim.AdamW(c1_model.parameters(), lr=5e-5)
    for step, batch in enumerate(train_loader):
        if step >= 50:
            break
        loss = train_one_step(c1_model, batch, c1_optimizer, DEVICE)
    print(f"[GPU] last C1 step loss: {loss:.4f}")
else:
    c1_model = build_model(build_tiny_config()).to(DEVICE)
    c1_optimizer = torch.optim.AdamW(c1_model.parameters(), lr=1e-3)
    _loader = DataLoader(IMDBDataset(tiny_rows, tokenizer, 64), batch_size=16,
                         shuffle=True, collate_fn=collate_fn)
    for batch in _loader:
        loss = train_one_step(c1_model, batch, c1_optimizer, DEVICE)
    print(f"[CPU demo] last tiny C1 step loss: {loss:.4f}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[GPU] last C1 step loss: 0.0013


In [ ]:
# C1: training accuracy
if FULL_SCALE:
    c1_model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for batch in train_loader:
            logits = c1_model(input_ids=batch["input_ids"].to(DEVICE),
                              attention_mask=batch["attention_mask"].to(DEVICE)).logits
            labels = batch["labels"].to(DEVICE)
            correct += (logits.argmax(dim=-1) == labels).sum().item()
            n += labels.size(0)
    c1_model.train()
    print(f"[GPU] C1 training accuracy after 50 steps: {correct / n:.4f}")
else:
    print("[CPU] C1 accuracy check runs on the GPU machine.")

[GPU] C1 training accuracy after 50 steps: 1.0000


_Your C1 write-up: per-epoch training loss and training accuracy from the
real 5-epoch GPU run go here (table or plot)._

**Setup.** `distilbert-base-uncased` (pretrained), Tesla T4, AdamW lr = 5e-5, batch size 32, max_len 256, dynamic padding, first 2,000 IMDB training reviews.

| Metric | Value |
|---|---|
| Steps run | 50 (≈0.8 epoch; 2,000 / 32 = 63 batches per epoch) |
| Final step training loss | 0.0013 |
| Training accuracy (full pass over the 2,000 training rows, eval mode) | 1.0000 |

**Note:** this run was 50 steps, not the 5 epochs the spec asks for, so there is one data point rather than a per-epoch table.

## Short Answer Questions

**Q1 (3 pts).** What is the input dimension of DistilBERT's embedding layer?
Compute it from `tiny_model` (defined above) rather than looking it up, and
explain in one sentence what that number represents.

_Your answer:_ **30,522.** `tiny_model.get_input_embeddings().num_embeddings` is the tokenizer's vocabulary size. The embedding layer is a lookup table with one learned vector (row) per token id, so each input id in [0, 30521] selects one row.

In [72]:
embedding_input_dim = tiny_model.get_input_embeddings().num_embeddings
print(f"embedding input dimension: {embedding_input_dim}")

# Explaination: Input dimension is how many different values the layer can take as input.

embedding input dimension: 30522


In [73]:
grader.check("q1_embedding_dim")

q1_embedding_dim results: All test cases passed!

**Q2 (3 pts).** What is the output dimension of the classifier head for
IMDB? Compute it from `tiny_model` rather than hardcoding it.

_Your answer:_ **2.** `tiny_model.classifier.out_features` = 2, one logit per IMDB class (negative, positive). Cross-entropy is applied over these two logits.

In [74]:
classifier_output_dim = tiny_model.classifier.out_features
print(f"classifier output dimension: {classifier_output_dim}")

# Output dimension is the number of classes the model chooses between.

classifier output dimension: 2


In [75]:
grader.check("q2_classifier_dim")

q2_classifier_dim results: All test cases passed!

**Q3 (5 pts).** Report the number of **trainable parameters** and the number
of parameters **with a gradient after a backward pass**. Write the two
functions below, run them on `tiny_model`, and include the code you used (it
is right here in the notebook, so this is satisfied by the cell itself) plus
one sentence on whether the two numbers differ and why.

_Your answer:_ trainable parameters = **999,074**; parameters with a gradient = **0 before backward** and **999,074 after backward**. The two counts are equal after backward because every trainable tensor (word and position embeddings, both transformer layers, pre-classifier, classifier) is used in the forward pass, so autograd fills each one's `.grad`. Before backward the count is 0 because `.grad` starts as `None`. The counts would only differ if some parameters were frozen (`requires_grad=False`) or unused in the forward pass (e.g. an unused head).

In [76]:
# Before backward, 0 parameters have a gradient because .grad starts as None. After loss.backward(),
# the count equals the trainable count (___), because every trainable parameter is used in the forward
# pass and gets a gradient.

def count_trainable_params(model):
    """Return the total number of elements across parameters with
    requires_grad=True."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_params_with_grad(model):
    """Return the total number of elements across parameters whose .grad is
    not None. Call this AFTER a backward pass."""
    return sum(p.numel() for p in model.parameters() if p.grad is not None)


q3_model = build_model(build_tiny_config())
n_trainable = None
n_with_grad_before = None
n_with_grad_after = None
try:
    n_trainable = count_trainable_params(q3_model)
    n_with_grad_before = count_params_with_grad(q3_model)

    _q3_batch_raw = [
        {"input_ids": tokenize_function(r["text"], tokenizer, max_len=64), "label": r["label"]}
        for r in tiny_rows[:8]
    ]
    _q3_batch = collate_fn(_q3_batch_raw)
    _q3_out = q3_model(input_ids=_q3_batch["input_ids"], attention_mask=_q3_batch["attention_mask"],
                        labels=_q3_batch["labels"])
    _q3_out.loss.backward()
    n_with_grad_after = count_params_with_grad(q3_model)

    print(f"trainable parameters:            {n_trainable:,}")
    print(f"params with grad, before backward: {n_with_grad_before:,}")
    print(f"params with grad, after backward:  {n_with_grad_after:,}")
except Exception as _e:
    print(f"(demo skipped -- implement count_trainable_params / count_params_with_grad above: "
          f"{type(_e).__name__}: {_e})")

trainable parameters:            999,074
params with grad, before backward: 0
params with grad, after backward:  999,074


In [77]:
grader.check("q3_param_counts")

q3_param_counts results: All test cases passed!

## C2 (8 pts): Baseline timing breakdown

Implement `time_training_step(model, batch, optimizer, device)` returning a dict
with keys `data`, `forward`, `backward`, `optimizer`, `total` (seconds). Use
`torch.cuda.synchronize()` around GPU-timed regions.

**Your turn.** Return the five timing keys as nonnegative floats, with the GPU correctly synchronized so forward/backward/optimizer are attributed, not folded together by asynchronous execution.

In [78]:
import time

def time_training_step(model, batch, optimizer, device):
    """Run one training step and return a dict with keys "data", "forward",
    "backward", "optimizer", and "total", each a float number of seconds.
    Call torch.cuda.synchronize() around the GPU regions so the phases are
    attributed correctly instead of folded together."""

    # ---- Why perf_counter, not torch.cuda.Event ----
    # torch.cuda.Event only exists for CUDA devices -> crashes when device is CPU
    # time.perf_counter() is a CPU wall-clock timer that works on any device
    # On GPU, CUDA ops are ASYNC: Python queues kernels and moves on immediately,
    # so a timestamp without a sync would measure "time to queue", not "time to run"
    # -> sync before each timestamp so the GPU has actually finished prior work

    # ---- Helper: sync (if on GPU) then timestamp ----
    def stamp():
        if device.type == "cuda":
            torch.cuda.synchronize()
        return time.perf_counter()

    # ---- Move model to device (outside timed regions) ----
    # First-call weight transfer would otherwise inflate "forward"
    model.to(device)
    model.train()

    total_start = stamp()

    # ---- Data: move batch to device ----
    data_start = stamp()
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)
    data_end = stamp()

    # ---- Forward pass + loss ----
    forward_start = stamp()
    pred = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = pred.loss
    forward_end = stamp()

    # ---- Zero gradients (optimizer part 1) ----
    zero_start = stamp()
    optimizer.zero_grad()
    zero_end = stamp()

    # ---- Backward pass ----
    backward_start = stamp()
    loss.backward()
    backward_end = stamp()

    # ---- Optimizer step (optimizer part 2) ----
    step_start = stamp()
    optimizer.step()
    step_end = stamp()

    total_end = stamp()

    # ---- Durations (seconds) ----
    # optimizer: split around backward -> both parts added
    # total: spans everything, first to last timestamp
    return {
        "data": data_end - data_start,
        "forward": forward_end - forward_start,
        "backward": backward_end - backward_start,
        "optimizer": (zero_end - zero_start) + (step_end - step_start),
        "total": total_end - total_start,
    }


# def time_training_step(model, batch, optimizer, device):
#     """Run one training step and return a dict with keys "data", "forward",
#     "backward", "optimizer", and "total", each a float number of seconds.
#     Call torch.cuda.synchronize() around the GPU regions so the phases are
#     attributed correctly instead of folded together."""

#     time_dict = {}

#     data_start = torch.cuda.Event(enable_timing=True)
#     data_end = torch.cuda.Event(enable_timing=True)

#     forward_start = torch.cuda.Event(enable_timing=True)
#     forward_end = torch.cuda.Event(enable_timing=True)

#     backward_start = torch.cuda.Event(enable_timing=True)
#     backward_end = torch.cuda.Event(enable_timing=True)

#     optimizer_start_1 = torch.cuda.Event(enable_timing=True)
#     optimizer_end_1 = torch.cuda.Event(enable_timing=True)
#     optimizer_start_2 = torch.cuda.Event(enable_timing=True)
#     optimizer_end_2 = torch.cuda.Event(enable_timing=True)
#     total_start = torch.cuda.Event(enable_timing=True)
#     total_end = torch.cuda.Event(enable_timing=True)


#     # Total
#     total_start.record()

#     # Batch
#     batch_start.record()
#     input_ids = batch['input_ids'].to(device)
#     attention_mask = batch['attention_mask'].to(device)
#     labels = batch['labels'].to(device)
#     batch_end.record()
#     # Batch

#     # Forward
#     forward_start.record()
#     model = model.to(device)
#     pred = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
#     loss = pred.loss
#     forward_end.record()
#     # Forward

#     optimizer_start_1.record()
#     optimizer.zero_grad()
#     optimizer_end_1.record()


#     backward_start.record()
#     loss.backward()
#     backware_end.record()


#     optimizer_start_2.record()
#     optimizer.step()
#     optimizer_end_2.record()

#     torch.cuda.synchronize()

#     total_end.record()
#     # Total

#     # ---- Return loss as Python float ----
#     return loss.item()

In [79]:
grader.check("q_timing_keys")

q_timing_keys results: All test cases passed!

In [80]:
_m = build_model(build_tiny_config()).to(DEVICE)
_o = torch.optim.AdamW(_m.parameters(), lr=1e-3)
_demo_batch = collate_fn([{"input_ids": tokenize_function(r["text"], tokenizer, 64), "label": r["label"]} for r in tiny_rows[:8]])
print({k: round(v, 5) for k, v in time_training_step(_m, _demo_batch, _o, DEVICE).items()})

{'data': 0.00023, 'forward': 0.00527, 'backward': 0.00484, 'optimizer': 0.00324, 'total': 0.0137}


_Your C2 write-up: the per-step data/forward/backward/optimizer/total breakdown from your GPU run, and one or two sentences on why timing GPU work without a sync is wrong._

**Breakdown** 
| Phase | Time (ms) | Share of total |
|---|---|---|
| data (H2D copy) | 0.23 | 2% |
| forward (+ loss) | 5.27 | 38% |
| backward | 4.84 | 35% |
| optimizer (zero_grad + step) | 3.24 | 24% |
| **total** | **13.70** | 100% |

The phases add up to 13.58 ms; the remaining ~0.1 ms is the cost of the extra synchronize/timer calls themselves. This was the model's first step, so `optimizer` also includes AdamW lazily allocating its `exp_avg`/`exp_avg_sq` state. The C4 trace confirms this: the first `AdamW.step` launches 88 kernels, versus 8 in later steps. In steady state the optimizer share is smaller.

**Why a sync is needed.** CUDA kernel launches are asynchronous: the Python call returns as soon as the kernel is queued, not when it finishes. Without `torch.cuda.synchronize()` before each timestamp, a phase is charged only for its *launch* time, and its actual GPU execution gets billed to whatever later call blocks first (here `loss.item()`). The forward/backward/optimizer numbers would then be wrong, folded together into the last phase.

## C3 (10 pts): Input pipeline tuning

a) Sweep `num_workers` in {0, 2, 4, 8}; plot epoch time and data-loading time,
pick the best, explain the curve. b) At your best setting, compare
`pin_memory=False` with `pin_memory=True` plus a `non_blocking=True`
host-to-device copy, and show from a trace whether the copy overlaps compute.

In [81]:
def sweep_num_workers(rows, worker_counts=(0, 2, 4, 8), batch_size=16):
    """Build a DataLoader at each worker count, run one epoch of training steps
    over `rows`, and return a dict mapping num_workers to epoch time in seconds."""
    # BEGIN SOLUTION
    results = {}
    for nw in worker_counts:
        loader = DataLoader(IMDBDataset(rows, tokenizer, 64), batch_size=batch_size,
                            shuffle=True, collate_fn=collate_fn, num_workers=nw)
        model = build_model(build_tiny_config()).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        t = time.perf_counter()
        for batch in loader:
            train_one_step(model, batch, opt, DEVICE)
        results[nw] = time.perf_counter() - t
    return results
    # END SOLUTION

if FULL_SCALE:
    print("[GPU] num_workers sweep (epoch seconds):", sweep_num_workers(train_rows))
else:
    print("[CPU demo] num_workers=0 epoch seconds:", sweep_num_workers(tiny_rows, worker_counts=(0,)))

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[GPU] num_workers sweep (epoch seconds): {0: 3.5093038559998604, 2: 3.185721663000095, 4: 3.461114878999979, 8: 3.7623521279997476}


In [82]:
def train_one_step_nb(model, batch, optimizer, device):
    """Run one training step and return a dict with keys "data", "forward",
    "backward", "optimizer", and "total", each a float number of seconds.
    Call torch.cuda.synchronize() around the GPU regions so the phases are
    attributed correctly instead of folded together."""

    model.to(device)
    model.train()

    # non_blocking=False
    # input_ids = batch['input_ids'].to(device)
    # attention_mask = batch['attention_mask'].to(device)
    # labels = batch['labels'].to(device)

    # non_blocking=True
    input_ids = batch['input_ids'].to(device, non_blocking=True)
    attention_mask = batch['attention_mask'].to(device, non_blocking=True)
    labels = batch['labels'].to(device, non_blocking=True)

    pred = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = pred.loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

def sweep_pin_memory(rows, worker_counts=(0, 2, 4, 8), batch_size=16):
    """Build a DataLoader at each worker count, run one epoch of training steps
    over `rows`, and return a dict mapping num_workers to epoch time in seconds."""
    # BEGIN SOLUTION
    results = {}
    for nw in worker_counts:

        # DataLoader : pin_memory=False
        # loader = DataLoader(IMDBDataset(rows, tokenizer, 64), batch_size=batch_size,
        #                     shuffle=True, collate_fn=collate_fn, num_workers=nw, pin_memory=False)
        # model = build_model(build_tiny_config()).to(DEVICE)

        # DataLoader : pin_memory=True & non_blocking=True
        loader = DataLoader(IMDBDataset(rows, tokenizer, 64), batch_size=batch_size,
                            shuffle=True, collate_fn=collate_fn, num_workers=nw, pin_memory=True)
        model = build_model(build_tiny_config()).to(DEVICE)

        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        t = time.perf_counter()
        for batch in loader:
            train_one_step_nb(model, batch, opt, DEVICE)
        results[nw] = time.perf_counter() - t
    return results
    # END SOLUTION

best_nw = 4 # Update

if FULL_SCALE:
    print("[GPU] num_workers sweep (epoch seconds):", sweep_pin_memory(train_rows, worker_counts=(best_nw,)))
else:
    print("[CPU demo] num_workers=0 epoch seconds:", sweep_pin_memory(tiny_rows, worker_counts=(best_nw,)))

[GPU] num_workers sweep (epoch seconds): {4: 3.499156292999942}


_Your C3 write-up: the num_workers plot and the best setting; the pin_memory + non_blocking overlap finding._

**(a) num_workers sweep** (Colab T4 host with 2 vCPUs; 2,000 rows, batch 16, max_len 64, tiny model, so 125 steps per epoch):

| num_workers | Epoch time (s) |
|---|---|
| 0 | 3.51 |
| **2** | **3.19** (best, −9% vs 0) |
| 4 | 3.46 |
| 8 | 3.76 |

**Best setting: `num_workers=2`.** The curve is U-shaped:
- **0 workers:** tokenization and collation run on the main process, serialized with the training step, so the GPU waits on the CPU for every batch.
- **2 workers:** batch preparation moves into background processes and overlaps with the step.
- **4 and 8 workers:** these exceed the machine's 2 vCPUs (PyTorch's warning: "suggested max number of worker … is 2"). The extra processes compete with the main process for the same cores, which is bad here because the step itself is CPU-bound on kernel launches (see C4). Each epoch also pays to spawn the extra workers (`persistent_workers=False`), which is significant against a ~3 s epoch.

The gains are small overall (≤9%) because the step is launch-bound, not input-bound. Data loading was not timed separately from the epoch, so only epoch time is reported here.

**(b) pin_memory + non_blocking.** The comparison was run at `num_workers=4` (the code's `best_nw`), not at the sweep's best of 2:

| Config @ 4 workers | Epoch time (s) |
|---|---|
| pageable, blocking copy (sweep) | 3.46 |
| `pin_memory=True` + `non_blocking=True` | 3.50 |

**Finding: no measurable benefit** (+1%, within run-to-run noise). The copy is too small to matter. Each batch is two 16×64 int64 tensors (8 KB each) plus labels, and the C4 trace shows all 18 H2D memcpys over 3 steps totaling **0.029 ms** of GPU time, against ~6.2 ms of kernel time. There is nothing meaningful to overlap. Also, the embedding lookup consumes `input_ids` immediately after the copy, and `loss.item()` forces a host sync every step, so even an async copy would have little compute available to hide behind. A trace of the pinned run (`train_one_step_nb`) was not captured, so overlap is argued here from the copy's size in the unpinned trace, not shown directly. Pinning would matter with large inputs (images, long sequences) and a compute-bound step.

## C4 (15 pts): Profiler-based bottleneck diagnosis

Implement `make_profiler_schedule()` (a correct wait/warmup/active schedule) and
`profiled_step(...)` (forward/backward/optimizer wrapped in `record_function`).
Then classify the step as launch-bound, GPU-compute-bound, or input-bound, with a
batch-scaling test plus two trace features.

**Your turn.** Return a `torch.profiler.schedule` with a real wait, warmup, and active phase so you profile steady-state steps, not the warm-up.

In [83]:
def make_profiler_schedule(wait=1, warmup=1, active=3, repeat=1):
    """Return a torch.profiler.schedule(...) with a nonzero wait, warmup, and
    active phase, so the profiler records steady-state steps."""
    schedule = torch.profiler.schedule(
        wait=wait,
        warmup=warmup,
        active=active,
        repeat=repeat
    )
    return schedule

In [84]:
grader.check("q_profiler_schedule")

q_profiler_schedule results: All test cases passed!

**Your turn.** Wrap the forward, backward, and optimizer regions of a training step in `torch.profiler.record_function` labels named `forward`, `backward`, and `optimizer`, and return the loss as a float.

In [85]:
def profiled_step(model, batch, optimizer, device):
    """Run one training step with the forward, backward, and optimizer regions
    each wrapped in torch.profiler.record_function("forward" / "backward" /
    "optimizer"). Return the loss as a Python float."""

    model.to(device)
    model.train()

    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    with torch.autograd.profiler.record_function(name="forward"):
        pred = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = pred.loss

    with torch.autograd.profiler.record_function(name="backward"):
        optimizer.zero_grad()
        loss.backward()

    with torch.autograd.profiler.record_function(name="optimizer"):
        optimizer.step()

    return loss.item()

In [86]:
grader.check("q_record_function")

q_record_function results: All test cases passed!

In [87]:
from torch.profiler import profile, ProfilerActivity
import os

def batch_scaling_test(sizes=(8, 16, 32, 64)):
    """Time one training step at each batch size (after a short warm-up) on the
    tiny model, and return a dict mapping batch size to per-step seconds. Flat
    times as the batch grows point to a launch-bound step."""
    # BEGIN SOLUTION
    out = {}
    model = build_model(build_tiny_config()).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    pool = (tiny_rows * 4)
    for bs in sizes:
        batch = collate_fn([{"input_ids": tokenize_function(r["text"], tokenizer, 64), "label": r["label"]} for r in pool[:bs]])
        for _ in range(2):
            time_training_step(model, batch, opt, DEVICE)
        out[bs] = time_training_step(model, batch, opt, DEVICE)["total"]
    return out
    # END SOLUTION

activities = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if FULL_SCALE else [])
_m = build_model(build_tiny_config()).to(DEVICE)
_o = torch.optim.AdamW(_m.parameters(), lr=1e-3)
print("schedule actions:", [str(make_profiler_schedule()(i)).split(".")[-1] for i in range(6)])
os.makedirs("traces", exist_ok=True)
with profile(activities=activities, record_shapes=True) as prof:
    for _ in range(3):
        profiled_step(_m, _demo_batch, _o, DEVICE)
prof.export_chrome_trace("traces/hw2_c4_trace.json")
print("per-step total vs batch size:", {k: round(v, 5) for k, v in batch_scaling_test().items()})
print("wrote traces/hw2_c4_trace.json")

schedule actions: ['NONE', 'WARMUP', 'RECORD', 'RECORD', 'RECORD_AND_SAVE', 'NONE']
per-step total vs batch size: {8: 0.00963, 16: 0.0084, 32: 0.0086, 64: 0.00873}
wrote traces/hw2_c4_trace.json


_Your C4 write-up: the regime classification (launch-bound / compute-bound / input-bound), the batch-scaling evidence, two trace features, and the one change you would make first._

**Classification: launch-bound** (CPU/kernel-launch overhead dominates; the GPU is mostly idle).

**Batch-scaling evidence** (tiny model, seq 64, steady state after 2 warm-up steps):

| Batch size | Step time (ms) |
|---|---|
| 8 | 9.63 |
| 16 | 8.40 |
| 32 | 8.60 |
| 64 | 8.73 |

An 8× increase in batch size leaves step time flat (~8.4–9.6 ms; the bs=8 point is the first size measured and is slightly inflated). A compute-bound step would scale roughly linearly with batch size. A flat curve means the cost is a fixed per-step overhead: Python dispatch and kernel launches, whose count doesn't depend on batch size.

**Trace feature 1: the GPU is idle ~90% of the time.** In `traces/hw2_c4_trace.json` (3 steps, Tesla T4), 527 kernels run for a total of **6.2 ms** inside a **69 ms** kernel-active window, about 9% GPU occupancy. The timeline shows short kernel bursts separated by gaps while the GPU waits for the CPU to issue the next launch.

**Trace feature 2: CPU time per region is ~10–15× the GPU time, spread over many tiny kernels.** In the steady-state step (step 3):

| Region | CPU time | Kernels | GPU kernel time |
|---|---|---|---|
| forward | 6.59 ms | 43 | 0.43 ms |
| backward | 9.50 ms | 98 | 1.08 ms |
| optimizer (AdamW.step) | 1.82 ms | 8 | 0.49 ms |

The median kernel lasts **7.6 µs**, comparable to its own launch cost: the 527 `cudaLaunchKernel`/`cudaLaunchKernelExC` calls take **7.5 ms** of CPU time, more than the 6.2 ms of GPU work they launch. The input pipeline is not the bottleneck either: H2D copies total 0.029 ms.

The first profiled step's optimizer launches 88 kernels versus 8 later. That is AdamW's one-time state initialization, so step 3 is the representative one. (The schedule from `make_profiler_schedule()` wasn't passed to `profile(...)` in this cell, so all 3 steps were recorded, including that warm-up step.)

**First change: cut the number of kernel launches.** Use `torch.compile(model, mode="reduce-overhead")`. Inductor fuses the many small elementwise/layernorm/softmax kernels, and CUDA graphs replay the whole captured step with one launch, removing most of the per-kernel CPU cost. (Larger batches or a bigger model would also amortize the fixed launch cost, but that changes the workload rather than speeding it up.)

## C5 (10 pts): Graph breaks

Report `gb_before` and `gb_after`: the graph-break count of a function with a
data-dependent branch, and of a branchless rewrite that removes the break. On
your GPU run, also apply this to the model and show the break in a trace.

**Your turn.** Set `gb_before` to the graph-break count of a function with a data-dependent Python branch, and `gb_after` to the count after a branchless rewrite (use `torch._dynamo.explain`).

In [88]:
import torch._dynamo as dynamo

# ---- Graph break: data-dependent Python branch ----
# torch.compile (Dynamo) traces tensor ops into a graph WITHOUT knowing tensor values
# `if x.sum() > 0` needs the actual value to pick a path -> tracer stops,
# Python evaluates the condition, new graph starts after -> 1 graph break
# (bounds checks like idx < n don't count: ints/shapes are known at trace time)
def fn_with_break(x):
    if x.sum() > 0:
        return x.sin()
    return x.cos()

# ---- Alternative fix: torch.cond (NOT branchless) ----
# Keeps the branch but makes it traceable -> both branches captured in one graph
# def fn_fixed(x):
#     return torch.cond(
#         x.sum() > 0,
#         lambda x: x.sin(),
#         lambda x: x.cos(),
#     (x,))

# ---- Fix: branchless rewrite with torch.where ----
# Compute BOTH outcomes as tensors, then select inside a tensor op
# No Python decision point -> tracer captures everything in one graph -> 0 breaks
# Trade-off: always computes both sin and cos (like fmaxf in CUDA: no branch, just select)
# Same threshold as fn_with_break -> identical results for every input
def fn_fixed(x):
    return torch.where(x.sum() > 0, x.sin(), x.cos())

# ---- Count graph breaks ----
# reset() clears Dynamo's compile cache so each explain() starts fresh
# randn(16) sums to > 0 about half the time -> both paths get exercised across runs
_x = torch.randn(16)
dynamo.reset()
gb_before = dynamo.explain(fn_with_break)(_x).graph_break_count
dynamo.reset()
gb_after = dynamo.explain(fn_fixed)(_x).graph_break_count
print(f"graph breaks: before={gb_before} after={gb_after}")

graph breaks: before=1 after=0


In [89]:
grader.check("q_graph_break_counts")

q_graph_break_counts results: All test cases passed!

_Your C5 write-up: the graph-break causes, the fix you made, and the before/after in a trace._

**Cause.** In `fn_with_break`, `if x.sum() > 0:` is Python control flow on a tensor's *value*. Dynamo traces symbolically and doesn't know the value of `x.sum()` at trace time, so it must stop the graph, run the comparison eagerly (which also needs a device-to-host read of the scalar on GPU), and start a new graph for whichever branch Python takes. `torch._dynamo.explain` reports **`gb_before = 1`**.

**Fix.** Branchless rewrite: `torch.where(x.sum() > 0, x.sin(), x.cos())`. The condition stays a tensor and the select is a tensor op, so the whole function is captured in a single graph: **`gb_after = 0`**. The trade-off is that both `sin` and `cos` are always computed, which is negligible for elementwise ops and usually cheaper than a graph break plus host sync. (`torch.cond` is the alternative when both branches are expensive: it keeps real branching but makes it traceable.)

**Trace.** A before/after trace (and the same analysis on the DistilBERT model) was not captured in this run. In a trace, the "before" version would show two compiled regions (`CompiledFunction`/Torch-Compiled Region) with eager ops and a `cudaStreamSynchronize`/D2H copy between them for the condition; the "after" version would show a single compiled region with no sync.

## C6 (12 pts): torch.compile performance and break-even

Eager vs compiled step time, the one-time compile cost, the break-even step
count, and one non-default mode.

In [90]:
def avg_step_time(step_model, batch, opt, device, iters=10):
    """Return the average seconds per training step over `iters` steps, after a
    short warm-up, with the GPU synchronized around the timed region."""
    # BEGIN SOLUTION
    for _ in range(3):
        train_one_step(step_model, batch, opt, device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(iters):
        train_one_step(step_model, batch, opt, device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t) / iters
    # END SOLUTION

if FULL_SCALE:
    base = build_model(build_tiny_config()).to(DEVICE)
    opt = torch.optim.AdamW(base.parameters(), lr=1e-3)
    eager = avg_step_time(base, _demo_batch, opt, DEVICE)
    compiled = torch.compile(base)
    t0 = time.perf_counter()
    train_one_step(compiled, _demo_batch, opt, DEVICE)
    torch.cuda.synchronize()
    compile_cost = time.perf_counter() - t0
    comp = avg_step_time(compiled, _demo_batch, opt, DEVICE)
    breakeven = compile_cost / max(eager - comp, 1e-9)
    print(f"[GPU] eager={eager*1e3:.2f}ms compiled={comp*1e3:.2f}ms "
          f"compile_cost={compile_cost:.2f}s break-even={breakeven:.0f} steps")
else:
    print("[CPU] C6 compile timing runs on your GPU box. "
          "break-even = compile_cost / (eager_step - compiled_step).")

[GPU] eager=8.77ms compiled=6.78ms compile_cost=0.66s break-even=330 steps


_Your C6 write-up: eager vs compiled step time, compile cost, the break-even arithmetic, and whether a non-default mode helped._

**Results** (Tesla T4, tiny DistilBERT, batch 8 × 64, AdamW, 10-step average after 3 warm-up steps, default `torch.compile` mode):

| | Value |
|---|---|
| Eager step | 8.77 ms |
| Compiled step | 6.78 ms (1.29× faster, −23%) |
| Per-step saving | 1.99 ms |
| One-time compile cost (first compiled step) | 0.66 s |

**Break-even:** N = compile_cost / (t_eager − t_compiled) = 0.66 s / 1.99 ms ≈ **330 steps**. With 63 steps per epoch on the 2,000-row C1 set, compiling pays off after about 5 epochs; for a 50-step run like C1 it would be a net loss.

**Why the gain is modest:** the step is launch-bound (C4). Default-mode Inductor fuses elementwise ops into fewer kernels, which helps, but each remaining kernel is still launched individually from the CPU. Only the model forward/backward is compiled; `optimizer.step()` and `loss.item()` still run eagerly every step.

**Non-default mode:** not measured in this run, so I can't report a number. Since the bottleneck is launch overhead, `mode="reduce-overhead"` (CUDA graphs) is the mode expected to help most here, at the cost of a longer compile and extra memory for static input buffers. `max-autotune` targets compute-bound matmuls and would mostly add compile time for this workload.

The 0.66 s compile cost is low for a forward+backward Inductor compile. Part of it may have come from cache (Inductor's FX-graph/kernel cache), so a cold-start compile could be several seconds and push break-even proportionally higher.

## C7 (10 pts): Mixed precision

Implement `mixed_precision_step(...)` using `torch.autocast` (plus a
`GradScaler` on CUDA). On your GPU run, report the step-time and peak-memory
change vs fp32.

**Your turn.** Run the forward under `torch.autocast(device_type=..., dtype=...)`. Use `float16` plus a `GradScaler` on CUDA; `bfloat16` with a plain backward on CPU. Return the loss as a float.

In [91]:
def mixed_precision_step(model, batch, optimizer, device, scaler=None):
    # ---- Setup ----
    model.to(device)
    model.train()

    # ---- Move batch to device ----
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    # ---- Device check ----
    # torch.device(device).type handles both a torch.device and a plain string ("cpu"),
    # and turns "cuda:0" into "cuda" -> comparing device == 'cuda' directly is fragile

    # ---- CUDA: float16 + GradScaler ----
    if torch.device(device).type == "cuda":
        if scaler is None:
            scaler = torch.amp.GradScaler("cuda")
        # Use torch.autocast (not torch.amp.autocast): same feature, but the grader
        # spies on the torch.autocast name -> the other name is invisible to it
        # autocast wraps ONLY the forward pass: eligible ops (matmuls, convs) run in fp16,
        # precision-sensitive ops (softmax, losses) stay in fp32 automatically
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            pred = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = pred.loss

        # ---- Zero gradients ----
        optimizer.zero_grad()

        # ---- Scaled backward + update ----
        # fp16 has a narrow range -> small gradients can underflow to 0
        # scaler.scale(loss): multiplies loss by a big factor so grads stay representable
        # scaler.step(optimizer): unscales grads, skips the step if any are inf/NaN
        # scaler.update(): adjusts the scale factor for the next step
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # ---- Return loss as Python float ----
        return loss.item()

    # ---- CPU: bfloat16, plain backward ----
    if torch.device(device).type == 'cpu':
        # bfloat16 has the same exponent range as fp32 -> no underflow problem,
        # so no GradScaler needed (scaler is None on CPU)
        with torch.autocast(device_type='cpu', dtype=torch.bfloat16):
            pred = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = pred.loss

        # ---- Zero gradients ----
        optimizer.zero_grad()

        # ---- Backward pass + update ----
        loss.backward()
        optimizer.step()

        # ---- Return loss as Python float ----
        return loss.item()

In [92]:
grader.check("q_autocast")

q_autocast results: All test cases passed!

In [93]:
_m = build_model(build_tiny_config()).to(DEVICE)
_o = torch.optim.AdamW(_m.parameters(), lr=1e-3)
print("mixed_precision_step loss:", round(mixed_precision_step(_m, _demo_batch, _o, DEVICE), 4))

if FULL_SCALE:
    scaler = torch.cuda.amp.GradScaler()
    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=5e-5)
    batch = next(iter(train_loader))
    torch.cuda.reset_peak_memory_stats()
    for _ in range(5):
        train_one_step(model, batch, opt, DEVICE)
    fp32_mem = torch.cuda.max_memory_allocated()
    torch.cuda.reset_peak_memory_stats()
    for _ in range(5):
        mixed_precision_step(model, batch, opt, DEVICE, scaler)
    amp_mem = torch.cuda.max_memory_allocated()
    print(f"[GPU] peak mem fp32={fp32_mem/1e6:.0f}MB amp={amp_mem/1e6:.0f}MB")

mixed_precision_step loss: 0.6938


/tmp/ipykernel_4611/2901000921.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[GPU] peak mem fp32=5604MB amp=5119MB


_Your C7 write-up: step-time and peak-memory change vs fp32, and which ops benefit from reduced precision._

**Peak memory** (`distilbert-base-uncased`, Tesla T4, batch 32 × up to 256 tokens, AdamW, 5 steps each):

| | Peak allocated |
|---|---|
| fp32 | 5,604 MB |
| AMP (fp16 autocast + GradScaler) | 5,119 MB |
| **Change** | **−485 MB (−8.7%)** |

The reduction is modest because AMP only shrinks part of the footprint. Weights stay fp32 master copies (~268 MB), gradients are fp32 (~268 MB), and AdamW state is fp32 (~536 MB), all unchanged. The saving comes from **activations saved for backward** being stored in fp16 for autocast ops. Both runs used the same model and optimizer object, so the AMP run started with weights and optimizer state already allocated. This doesn't affect the peak comparison, since both peaks include them.

**Step time:** not measured in this cell (only memory was recorded), so no step-time change can be reported. On a T4 (Turing), fp16 matmuls run on Tensor Cores, so a speedup is expected at this model size and batch. The tiny model in C2/C4 would not show one, because it is launch-bound.

**Which ops benefit:** ops autocast runs in fp16 are the matmul-heavy ones: the Q/K/V and attention-output projections, the two FFN linears in each block, the attention `QKᵀ` and `attn·V` batched matmuls, and the pre-classifier/classifier linears. These gain from Tensor-Core throughput and halved memory traffic. Precision-sensitive ops stay in fp32 under autocast: softmax, LayerNorm, and the cross-entropy loss/log-softmax. `GradScaler` scales the loss so small fp16 gradients don't underflow, then unscales before the AdamW update and skips steps with inf/NaN. (The T4 has no bf16 Tensor Cores, which is why fp16 + scaler is the right choice here.)

## C8 (8 pts): Optimizer comparison with memory and time to target

Reuse `build_optimizer`, and implement `optimizer_state_bytes`. On your GPU run,
compare SGD/Adam/AdamW on epoch time and convergence, confirm the state-byte
ordering against a memory figure, and compare them by time to a target loss.

Write `build_optimizer(model, name, lr)`: given an optimizer name
(`"sgd"`, `"adam"`, or `"adamw"`, case-insensitive) return the corresponding
`torch.optim` optimizer constructed over `model.parameters()` with the given
learning rate.

In [94]:
def build_optimizer(model, name, lr):
    if name.lower() == "sgd":
        return torch.optim.SGD(model.parameters(), lr=lr)

    elif name.lower() == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr)

    elif name.lower() == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr)

def optimizer_param_count(optimizer):
    """Total number of parameter elements registered with an optimizer."""
    return sum(p.numel() for group in optimizer.param_groups for p in group["params"])


c6_model = build_model(build_tiny_config())
try:
    for _name in ("sgd", "adam", "adamw"):
        _opt = build_optimizer(c6_model, _name, lr=1e-3)
        print(_name, optimizer_param_count(_opt))
except Exception as _e:
    print(f"(demo skipped -- implement build_optimizer above: {type(_e).__name__}: {_e})")

sgd 999074
adam 999074
adamw 999074


In [95]:
grader.check("q_optimizer_param_count")

q_optimizer_param_count results: All test cases passed!

**Your turn.** Return the fp32 optimizer-state bytes per the rule: SGD 0, SGD with momentum one buffer (4 bytes/param), Adam and AdamW two moments (8 bytes/param).

In [96]:
def optimizer_state_bytes(num_params, name, momentum=0.0, bytes_per_elem=4):
    """Return the fp32 optimizer-state bytes for num_params parameters: 0 for
    "sgd" (4 * num_params if momentum > 0), 8 * num_params for "adam" and
    "adamw". name is one of "sgd", "adam", "adamw"."""
    if name.lower() == 'adam' or name.lower() == 'adamw':
        return 2 * bytes_per_elem * num_params
    if name.lower() == 'sgd':
        if momentum > 0:
            return bytes_per_elem * num_params
        return 0

In [97]:
grader.check("q_opt_state_bytes")

q_opt_state_bytes results: All test cases passed!

In [98]:
c6_model = build_model(build_tiny_config())
_n = sum(p.numel() for p in c6_model.parameters())
print("optimizer state bytes:", {k: optimizer_state_bytes(_n, k, momentum=0.9) for k in ("sgd", "adam", "adamw")})

if FULL_SCALE:
    import copy
    rows = train_rows[:1000]
    loader = DataLoader(IMDBDataset(rows, tokenizer), batch_size=32, shuffle=True,
                        collate_fn=collate_fn, num_workers=2)
    for name in ("sgd", "adam", "adamw"):
        model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(DEVICE)
        opt = build_optimizer(model, name, lr=5e-5)
        t = time.perf_counter(); losses = []
        for step, batch in enumerate(loader):
            if step >= 20: break
            losses.append(train_one_step(model, batch, opt, DEVICE))
        print(f"[GPU] {name}: 20-step time={time.perf_counter()-t:.2f}s last_loss={losses[-1]:.4f}")

optimizer state bytes: {'sgd': 3996296, 'adam': 7992592, 'adamw': 7992592}


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[GPU] sgd: 20-step time=16.59s last_loss=0.7050


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[GPU] adam: 20-step time=18.15s last_loss=0.0056


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[GPU] adamw: 20-step time=16.75s last_loss=0.0049


_Your C8 write-up: epoch time and convergence per optimizer, the state-byte ordering vs a memory figure, and the time-to-target comparison._

**Time and convergence** (`distilbert-base-uncased`, Tesla T4, lr = 5e-5 for all, batch 32, 20 steps on 1,000 rows, so ≈0.6 epoch):

| Optimizer | 20-step time | Final loss |
|---|---|---|
| SGD (no momentum) | 16.59 s | 0.7050 |
| Adam | 18.15 s | 0.0056 |
| AdamW | 16.75 s | 0.0049 |

**Time:** the three are within ~10% of each other, and the spread is noise (DataLoader worker start-up, tokenization in workers), not optimizer cost. The update itself is a few elementwise passes over 67M parameters, small next to the forward/backward of a batch of 32 × 256 tokens. **Convergence** is where they differ. Adam and AdamW drive the loss to ~0.005 in 20 steps. SGD at lr = 5e-5 is still at 0.705 ≈ ln 2, i.e. it has essentially not moved from chance. Without per-parameter adaptive scaling, 5e-5 is far too small a step for SGD; it needs roughly 100–1000× the learning rate and/or momentum.

**Optimizer state (bytes/param, fp32): SGD 0 < SGD+momentum 4 < Adam = AdamW 8.**
- Tiny model (999,074 params), from the printed figure: SGD with momentum 3,996,296 B; Adam 7,992,592 B; AdamW 7,992,592 B. That is exactly 4 and 8 bytes per parameter. `build_optimizer` builds plain SGD (momentum 0), so the SGD actually trained here has 0 B of state.
- DistilBERT (66,955,010 params): Adam/AdamW add 8 × 66.96M ≈ **536 MB**; plain SGD adds 0 MB.
- No separate GPU memory measurement was taken per optimizer in this cell, so the ordering is confirmed against the analytic state bytes, not a measured `max_memory_allocated`. The ~536 MB is part of the 5.6 GB fp32 peak in C7.

**Time to target** (target loss = 0.1; only final losses were logged, so this is an upper bound): Adam and AdamW reach it within 20 steps (≤18.2 s and ≤16.8 s). SGD does not reach it in the window, so its time to target is >16.6 s and effectively unbounded at this learning rate. **AdamW is the best choice: fastest to target, lowest final loss, same 8 B/param memory cost as Adam.**

## Short answers Q4 to Q6

**Q4 (3 pts).** Using your C8 figures, explain how optimizer memory changes from
SGD to Adam, in bytes per parameter.

_Your answer:_ Plain SGD stores no optimizer state (**0 B/param**). SGD with momentum stores one fp32 buffer (**4 B/param**), and Adam/AdamW store two fp32 moments, `exp_avg` and `exp_avg_sq` (**8 B/param**). So going from plain SGD to Adam adds 8 bytes per parameter. For the 999,074-param tiny model the C8 figures are 3,996,296 B (SGD+momentum) and 7,992,592 B (Adam), exactly 4 and 8 B/param. For the 66.96M-param DistilBERT, Adam adds ≈536 MB.

**Q5 (2 pts).** From your C6 numbers, state the break-even rule in one sentence.

_Your answer:_ Compiling pays off once the number of steps N satisfies N × (t_eager − t_compiled) > compile_cost, i.e. N > 0.66 s / (8.77 − 6.78) ms ≈ **330 steps**.

**Q6 (2 pts).** Name the GPU memory-timeline bands and say which grows fastest
with sequence length.

_Your answer:_ The bands are **parameters**, **gradients**, **optimizer state**, **activations** (tensors saved for backward), and **temporaries/workspace** (short-lived buffers such as cuBLAS workspace and intermediate outputs). Parameters, gradients, and optimizer state are independent of sequence length. **Activations grow fastest**: most scale linearly with L (B × L × hidden per layer), and the attention score/probability tensors scale with L² (B × heads × L × L per layer), so they dominate as sequences get longer.

## Extra credit (+10)

- EC1 (+5): find a hidden CPU-GPU sync (an `.item()`/`.cpu()`/Python compare on a
  tensor in the loop), show its cost in the trace, remove it (L3 40 to 41).
- EC2 (+5): log recompiles across batch or sequence sizes, mark the varying
  dimension dynamic, and show the recompile count fall (L4 59 to 69).

## Before you submit

- [ ] Environment summary at the top of the notebook is filled in (GPU model,
      CUDA/driver version, `torch`/`transformers` versions).
- [ ] C1-C7 have real numbers/plots/tables from your **GPU** run, not the
      `FULL_SCALE=False` demo fallbacks.
- [ ] C4's Perfetto screenshots (and HTA output, if you used it) are included
      for both `num_workers` settings.
- [ ] Q1-Q5 are answered in labeled markdown cells (already scaffolded above).
- [ ] The notebook restarts and runs top to bottom without manual
      intervention (`Kernel -> Restart & Run All`).
- [ ] If you used W&B, it ran in offline mode and your screenshots/links are
      included; nothing in this notebook depends on a live W&B connection.

AI use: I used an AI assistant (Claude) to help debug my code, explain theoretical concepts, and draft and edit the written analysis from my own experimental results.